# Exploración de datos — MIT-BIH Arrhythmia Database

**Objetivo**: cargar, preprocesar y visualizar registros de ECG.  
Demostrar el pipeline completo de Fase 1: filtrado → normalización → detección de picos R → métricas HRV.

⚠️ **Aviso clínico**: este sistema es una herramienta de investigación/triaje.  
**No reemplaza la revisión de un cardiólogo.**

## 0. Entorno e imports

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Allow `from src.*` imports when notebook is inside notebooks/
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import wfdb

from src.io.loaders import load_wfdb_record
from src.preprocessing.filters import preprocess
from src.preprocessing.normalize import zscore_normalize
from src.features.rpeaks import detect_rpeaks, heart_rate
from src.features.hrv import compute_hrv_features
from src.visualization.plots import plot_ecg, plot_before_after

plt.rcParams.update({'figure.dpi': 120})
print('Imports OK')

## 1. Configuración

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'default.yaml'
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

DATA_DIR = PROJECT_ROOT / 'data' / 'mitdb'

print('Config cargado:')
print(yaml.dump(config, default_flow_style=False))

## 2. Descarga del dataset MIT-BIH

La descarga es **idempotente**: si los archivos ya existen, la celda lo detecta y se salta el paso.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Check for presence of at least the header for record 100
if not (DATA_DIR / '100.hea').exists():
    print('Descargando MIT-BIH Arrhythmia Database (~100 MB)...')
    wfdb.dl_database('mitdb', dl_dir=str(DATA_DIR))
    print('Descarga completada.')
else:
    print(f'Dataset ya presente en: {DATA_DIR}')

---
## 3. Registro 100 — Ritmo Sinusal Normal

In [ ]:
PATH_100 = str(DATA_DIR / '100')
signal_100_raw, fs_100, meta_100 = load_wfdb_record(PATH_100, channel=0)

print('=== Registro 100 — Metadata ===')
for k, v in meta_100.items():
    if k != 'annotations':
        print(f'  {k:<20s}: {v}')
n_ann = len(meta_100['annotations']) if meta_100['annotations'] else 0
print(f'  {"anotaciones":<20s}: {n_ann} eventos')

### 3.1 Preprocesamiento

In [ ]:
signal_100_filt = preprocess(signal_100_raw, fs_100, config['preprocessing'])
signal_100_norm = zscore_normalize(signal_100_filt)
print(f'Señal preprocesada — shape: {signal_100_norm.shape}, '
      f'rango: [{signal_100_norm.min():.2f}, {signal_100_norm.max():.2f}]')

### 3.2 Comparación antes / después (primeros 10 s)

In [ ]:
WIN = 10 * fs_100  # 10-second window
fig = plot_before_after(
    signal_100_raw[:WIN],
    signal_100_filt[:WIN],
    fs_100,
    title='Registro 100 (Ritmo Sinusal Normal) — Antes vs Después del Preprocesamiento',
)
plt.show()

### 3.3 Detección de picos R

In [ ]:
rpeaks_100, times_100 = detect_rpeaks(signal_100_norm, fs_100)
hr_100 = heart_rate(rpeaks_100, fs_100)

print(f'Picos R detectados : {len(rpeaks_100)}')
print(f'HR media           : {hr_100:.1f} bpm')

# Plot only the first 10 s window
mask_100 = rpeaks_100 < WIN
fig = plot_ecg(
    signal_100_norm[:WIN],
    fs_100,
    rpeaks=rpeaks_100[mask_100],
    title='Registro 100 — ECG Normalizado con Picos R',
)
plt.show()

### 3.4 Métricas HRV y mini-reporte

In [ ]:
features_100 = compute_hrv_features(
    rpeaks_100, fs_100,
    signal=signal_100_norm,
    min_sqi=config['quality']['min_sqi'],
)

def print_report(record_id: str, label: str, features: dict, meta: dict) -> None:
    sep = '=' * 58
    print(f'\n{sep}')
    print(f'  REGISTRO {record_id} | {label}')
    print(sep)
    print(f'  Duración       : {meta["duration_s"]:.1f} s')
    sqi_str = f"{features['sqi']:.3f}" if features['sqi'] is not None else 'N/A'
    print(f'  SQI            : {sqi_str}')
    print(f'  Analizable     : {"Sí" if features["analyzable"] else "No"}')

    if features['analyzable']:
        print(f'  HR             : {features["hr_bpm"]:.1f} bpm')
        print(f'  Mean RR        : {features["mean_rr"]:.1f} ms')
        print(f'  SDNN           : {features["sdnn"]:.1f} ms')
        print(f'  RMSSD          : {features["rmssd"]:.1f} ms')
        print(f'  pNN50          : {features["pnn50"]:.1f} %')

        if features['irregular']:
            print('  RR irregular   : *** POSIBLE IRREGULARIDAD (proxy FA) ***')
        else:
            print('  RR irregular   : No — ritmo regular')
    else:
        print('  [Señal de baja calidad — features HRV no calculados]')
    print(sep)

print_report('100', 'Ritmo Sinusal Normal', features_100, meta_100)

---
## 4. Registro 201 — Arritmia con irregularidad RR

> **Nota**: el registro 201 contiene bigeminismo ventricular y latidos
> de escape, lo que produce intervalos RR bimodales (corto-largo alternado)
> con RMSSD y CV elevados. Es el caso más próximo a irregularidad en el
> subconjunto mitdb; para FA clásica usar `afdb` en Fase 2.

In [ ]:
PATH_201 = str(DATA_DIR / '201')
signal_201_raw, fs_201, meta_201 = load_wfdb_record(PATH_201, channel=0)

print('=== Registro 201 — Metadata ===')
for k, v in meta_201.items():
    if k != 'annotations':
        print(f'  {k:<20s}: {v}')
n_ann = len(meta_201['annotations']) if meta_201['annotations'] else 0
print(f'  {"anotaciones":<20s}: {n_ann} eventos')

### 4.1 Preprocesamiento

In [ ]:
signal_201_filt = preprocess(signal_201_raw, fs_201, config['preprocessing'])
signal_201_norm = zscore_normalize(signal_201_filt)
print(f'Señal preprocesada — shape: {signal_201_norm.shape}')

### 4.2 Comparación antes / después (primeros 10 s)

In [ ]:
WIN_201 = 10 * fs_201
fig = plot_before_after(
    signal_201_raw[:WIN_201],
    signal_201_filt[:WIN_201],
    fs_201,
    title='Registro 201 (Arritmia) — Antes vs Después del Preprocesamiento',
)
plt.show()

### 4.3 Detección de picos R

In [ ]:
rpeaks_201, times_201 = detect_rpeaks(signal_201_norm, fs_201)
hr_201 = heart_rate(rpeaks_201, fs_201)

print(f'Picos R detectados : {len(rpeaks_201)}')
print(f'HR media           : {hr_201:.1f} bpm')

mask_201 = rpeaks_201 < WIN_201
fig = plot_ecg(
    signal_201_norm[:WIN_201],
    fs_201,
    rpeaks=rpeaks_201[mask_201],
    title='Registro 201 — ECG Normalizado con Picos R',
)
plt.show()

### 4.4 Métricas HRV y mini-reporte

In [ ]:
features_201 = compute_hrv_features(
    rpeaks_201, fs_201,
    signal=signal_201_norm,
    min_sqi=config['quality']['min_sqi'],
)
print_report('201', 'Arritmia (Bigeminismo Ventricular)', features_201, meta_201)

---
## 5. Tabla comparativa

In [ ]:
def _fmt(v, decimals=1):
    return f'{v:.{decimals}f}' if v is not None else 'N/A'

rows = [
    ('Registro',          '100', '201'),
    ('Ritmo',             'Sinusal Normal', 'Arritmia (201)'),
    ('HR (bpm)',          _fmt(features_100['hr_bpm']),  _fmt(features_201['hr_bpm'])),
    ('Mean RR (ms)',      _fmt(features_100['mean_rr']), _fmt(features_201['mean_rr'])),
    ('SDNN (ms)',         _fmt(features_100['sdnn']),    _fmt(features_201['sdnn'])),
    ('RMSSD (ms)',        _fmt(features_100['rmssd']),   _fmt(features_201['rmssd'])),
    ('pNN50 (%)',         _fmt(features_100['pnn50']),   _fmt(features_201['pnn50'])),
    ('SQI',              _fmt(features_100['sqi'], 3),  _fmt(features_201['sqi'], 3)),
    ('RR Irregular',     str(features_100['irregular']), str(features_201['irregular'])),
]

df = pd.DataFrame(rows, columns=['Métrica', 'Reg. 100', 'Reg. 201'])
df = df.set_index('Métrica')
df

---
## 6. Histograma de intervalos RR

Visualización adicional que muestra la distribución de los intervalos RR:
un ritmo sinusal debe mostrar una campana estrecha; una arritmia, una
distribución más amplia o bimodal.

In [ ]:
from src.features.hrv import compute_rr_intervals

rr_100 = compute_rr_intervals(rpeaks_100, fs_100)
rr_201 = compute_rr_intervals(rpeaks_201, fs_201)

fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=False)

for ax, rr, label, color in [
    (axes[0], rr_100, 'Reg. 100 — Sinusal Normal', '#1f77b4'),
    (axes[1], rr_201, 'Reg. 201 — Arritmia',       '#d62728'),
]:
    ax.hist(rr, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(rr), color='black', linestyle='--',
               linewidth=1.2, label=f'Mean={np.mean(rr):.0f} ms')
    ax.set_xlabel('RR interval (ms)')
    ax.set_ylabel('Count')
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.35)

fig.suptitle('Distribución de Intervalos RR', fontsize=12,
             fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()